# 04 — Period Detection

This notebook searches for the dominant pulsation period in the prepared photon arrival-time data.

## Objectives

- analyze the prepared arrival-time series;
- apply period-search logic;
- identify the most likely pulsation period;
- visualize the period-detection result.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.timeseries import LombScargle

plt.style.use('seaborn-v0_8-darkgrid')  
plt.rcParams.update({
    'figure.figsize': (10,6),
    'font.size': 14,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'lines.linewidth': 2,
    'lines.color': 'orangered',  
})


time = df_filtered['time'].values
flux = np.ones_like(time)  


min_freq = 0.1   
max_freq = 20.0  
frequency = np.linspace(min_freq, max_freq, 10000)

ls = LombScargle(time, flux)
power = ls.power(frequency)


idx_max = np.argmax(power)
dominant_freq = frequency[idx_max]
dominant_period = 1 / dominant_freq

print(f"Dominant frequency: {dominant_freq:.4f} Hz")
print(f"Corresponding period: {dominant_period:.4f} s")


plt.figure()
plt.plot(frequency, power, color='orangered')
plt.axvline(dominant_freq, color='darkgreen', linestyle='--', label=f'Dominant freq: {dominant_freq:.4f} Hz')
plt.xlabel('Frequency [Hz]')
plt.ylabel('Power')
plt.title('Lomb–Scargle Periodogram - Vela Pulsar')
plt.legend()
plt.show()


zoom_mask = (frequency > dominant_freq*0.9) & (frequency < dominant_freq*1.1)
plt.figure()
plt.plot(frequency[zoom_mask], power[zoom_mask], color='darkorange')
plt.axvline(dominant_freq, color='darkgreen', linestyle='--', label=f'Dominant freq')
plt.xlabel('Frequency [Hz]')
plt.ylabel('Power')
plt.title('Zoom around dominant frequency')
plt.legend()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.timeseries import LombScargle

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({
    'figure.figsize': (10,6),
    'font.size': 14,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'lines.linewidth': 2,
    'lines.color': 'orangered',  
})

hdul = fits.open('../data/raw/vela_photons.fits')
events = hdul[1].data

df = pd.DataFrame({
    "time": events["TIME"].astype('float64'),
    "energy": events["ENERGY"].astype('float64')
})

df_filtered = df[df["energy"] > 100]

print(f"Total events: {len(df)}")
print(f"Filtered events (>100 MeV): {len(df_filtered)}")

time = df_filtered['time'].values
flux = np.ones_like(time)  # для photon counting используем counts=1 на событие

min_freq = 0.1   # Гц
max_freq = 20.0  # Гц
frequency = np.linspace(min_freq, max_freq, 10000)

ls = LombScargle(time, flux)
power = ls.power(frequency)

idx_max = np.argmax(power)
dominant_freq = frequency[idx_max]
dominant_period = 1 / dominant_freq

print(f"Dominant frequency: {dominant_freq:.4f} Hz")
print(f"Corresponding period: {dominant_period:.4f} s")

plt.figure()
plt.plot(frequency, power, color='orangered')
plt.axvline(dominant_freq, color='darkgreen', linestyle='--', label=f'Dominant freq: {dominant_freq:.4f} Hz')
plt.xlabel('Frequency [Hz]')
plt.ylabel('Power')
plt.title('Lomb–Scargle Periodogram - Vela Pulsar')
plt.legend()
plt.show()


In [ ]:
import os
import matplotlib.pyplot as plt

images_dir = 'images'
os.makedirs(images_dir, exist_ok=True)

figures = [plt.figure(n) for n in plt.get_fignums()]

notebook_name = '03_period_detection'

for i, fig in enumerate(figures, start=1):
    filename = f"{notebook_name}_fig{i}.png"
    filepath = os.path.join(images_dir, filename)
    fig.savefig(filepath, dpi=150, bbox_inches='tight')
    print(f"Saved: {filepath}")

plt.close('all')


## Conclusion

The dominant pulsation period was identified from the prepared photon event data.  
This period can now be used to fold the signal and build the pulse profile.